# 04 · Archetype Characterisation
### Lifestyle Archetypes and Problematic Internet Use — Pipeline Notebook 4 of 5

**This notebook covers:**
10. Archetype characterisation (lifestyle profile, demographics, relation to PIU / mental health)
11. Visual summary (UMAP scatter, radar charts, heatmap)

**Loads:** `01_data_preparation.pkl`, `02_dimensionality_reduction_clustering.pkl`, `03_validation_stability.pkl`
**Produces:** `04_archetype_characterisation.pkl` — the final archetype-labelled dataset, available to `05_three_group_alternative_analysis.ipynb` for comparison.

**Reminder of the responsible-analysis boundary:** any outcome association reported below is **preliminary and subset-derived** (train-set labels only) and is reported as an association, not a validated clinical finding.

## Setup & Environment

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import f_oneway, chi2_contingency, kruskal


### Load artifacts from notebooks 01-03

In [ ]:
ARTIFACT_DIR = Path("/kaggle/working/artifacts")

with open(ARTIFACT_DIR / "01_data_preparation.pkl", "rb") as f:
    artifact_01 = pickle.load(f)

with open(ARTIFACT_DIR / "02_dimensionality_reduction_clustering.pkl", "rb") as f:
    artifact_02 = pickle.load(f)

with open(ARTIFACT_DIR / "03_validation_stability.pkl", "rb") as f:
    artifact_03 = pickle.load(f)

model_df = artifact_01["model_df"].copy()
CLUSTER_FEATURES = artifact_01["CLUSTER_FEATURES"]

X_scaled = artifact_02["X_scaled"]
embedding_2d = artifact_02["embedding_2d"]
best_k = artifact_02["best_k"]

FINAL_METHOD = artifact_03["FINAL_METHOD"]
FINAL_LABELS = artifact_03["FINAL_LABELS"]

print(f"Loaded model_df {model_df.shape}; final method = {FINAL_METHOD}")


## 10. Archetype Characterisation

We attach the selected cluster labels back to the modelling dataframe, and describe each archetype in terms of its lifestyle features, demographics, and - as a **preliminary, subset-derived association only** - its relationship to PIU and mental health screening scores.

In [ ]:
model_df["archetype"] = FINAL_LABELS
model_df["archetype"] = model_df["archetype"].astype("category")

print("Archetype sizes:")
display(model_df["archetype"].value_counts().sort_index())


In [ ]:
# --- lifestyle-feature profile per archetype ---
profile = model_df.groupby("archetype")[CLUSTER_FEATURES].agg(["mean", "std"])
display(profile.round(2))

# --- ANOVA / Kruskal-Wallis: do lifestyle features differ significantly by archetype? ---
print("\nSignificance of lifestyle-feature differences across archetypes:")
for feat in CLUSTER_FEATURES:
    groups = [g[feat].dropna().values for _, g in model_df.groupby("archetype")]
    f_stat, p_anova = f_oneway(*groups)
    h_stat, p_kw = kruskal(*groups)
    print(f"  {feat:32s}  ANOVA p={p_anova:.4f}   Kruskal-Wallis p={p_kw:.4f}")


In [ ]:
# --- sex distribution across archetypes (chi-square) ---
sex_table = pd.crosstab(model_df["archetype"], model_df["Basic_Demos-Sex"])
chi2, p_chi2, dof, _ = chi2_contingency(sex_table)
print("Sex distribution by archetype:")
display(sex_table)
print(f"Chi-square test: chi2={chi2:.2f}, dof={dof}, p={p_chi2:.4f}")

print("\nAge distribution by archetype:")
display(model_df.groupby("archetype")["Basic_Demos-Age"].describe().round(2))


### Relating archetypes to PIU and mental health screening scores

**Caution:** `PCIAT-PCIAT_Total`, `sii`, `CGAS-CGAS_Score` and `SDS-SDS_Total_T` were **not** used to form the archetypes above; they are examined here only to describe how the lifestyle-based segmentation relates to these screening outcomes. This is an **association from train-set labels on a subset of participants with usable actigraphy**, and should be read as hypothesis-generating, not as a validated clinical finding.

In [ ]:
outcome_vars = ["PCIAT-PCIAT_Total", "sii", "CGAS-CGAS_Score", "SDS-SDS_Total_T",
                "PreInt_EduHx-computerinternet_hoursday"]

fig, axes = plt.subplots(1, len(outcome_vars), figsize=(22, 4))
for ax, col in zip(axes, outcome_vars):
    sns.boxplot(data=model_df, x="archetype", y=col, ax=ax, palette="Set2")
    ax.set_title(col, fontsize=9)
plt.tight_layout()
plt.show()

print("Significance of outcome differences across archetypes (Kruskal-Wallis, robust to non-normal screening scores):")
for col in outcome_vars:
    groups = [g[col].dropna().values for _, g in model_df.groupby("archetype")]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        continue
    h_stat, p_val = kruskal(*groups)
    print(f"  {col:42s}  Kruskal-Wallis p={p_val:.4f}")

print("\nsii (Severity Impairment Index category) distribution by archetype:")
display(pd.crosstab(model_df["archetype"], model_df["sii"], normalize="index").round(2))


## 11. Visual Summary

UMAP scatter coloured by final archetype, plus one radar chart per archetype and a heatmap of standardised feature means - one clear summary visualisation per archetype.

In [ ]:
plt.figure(figsize=(7, 6))
scatter = plt.scatter(
    embedding_2d[:, 0], embedding_2d[:, 1],
    c=FINAL_LABELS, cmap="tab10", s=12, alpha=0.7
)
plt.legend(*scatter.legend_elements(), title="Archetype", loc="best")
plt.title("UMAP projection coloured by lifestyle archetype")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
plt.tight_layout()
plt.show()


In [ ]:
# standardised feature means per archetype, for the radar charts and heatmap
archetype_means_scaled = (
    pd.DataFrame(X_scaled.values, columns=CLUSTER_FEATURES, index=model_df.index)
    .groupby(model_df["archetype"])
    .mean()
)

def radar_chart(ax, values, labels, title):
    n = len(labels)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    values = list(values) + [values[0]]
    angles = angles + [angles[0]]
    ax.plot(angles, values, color="#4C72B0", linewidth=2)
    ax.fill(angles, values, color="#4C72B0", alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=7)
    ax.set_title(title, fontsize=11, pad=15)
    ax.set_yticklabels([])

n_arch = archetype_means_scaled.shape[0]
fig, axes = plt.subplots(1, n_arch, subplot_kw=dict(polar=True), figsize=(5 * n_arch, 5))
if n_arch == 1:
    axes = [axes]
for ax, (arch_id, row) in zip(axes, archetype_means_scaled.iterrows()):
    n_members = (model_df["archetype"] == arch_id).sum()
    radar_chart(ax, row.values, CLUSTER_FEATURES, f"Archetype {arch_id}  (n={n_members})")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, max(3, 0.6 * n_arch + 2)))
sns.heatmap(
    archetype_means_scaled, annot=True, fmt=".2f", cmap="vlag", center=0,
    cbar_kws={"label": "Standardised mean (z-score)"}
)
plt.title("Archetype profile heatmap (standardised lifestyle feature means)")
plt.ylabel("Archetype")
plt.tight_layout()
plt.show()


## Written Summary & Limitations

- **What was found:** a small number of lifestyle archetypes emerge from objective actigraphy features (sleep, activity intensity/variability, weekday-weekend consistency) combined with body-composition and self-reported activity measures. Archetypes differ significantly from one another on most of the input lifestyle features (see the ANOVA / Kruskal-Wallis results above).
- **Relationship to PIU / mental health:** any association between archetype membership and `PCIAT-PCIAT_Total`, `sii`, `CGAS-CGAS_Score` or `SDS-SDS_Total_T` is **preliminary and subset-derived** — it comes from train-set labels on the subset of participants with usable actigraphy, is not causal, and should be treated as hypothesis-generating rather than a validated clinical finding.
- **Sample restriction:** only participants with sufficient wearable-device wear time contribute to the clustering; this is not the full trial population, and non-wear is unlikely to be random (e.g. it may correlate with age, engagement, or the very lifestyle factors being studied), which is a limitation for how far these archetypes generalise.
- **Method sensitivity:** the chosen solution (printed as `FINAL_METHOD` / `best_k` in the cells above) was one of three algorithms compared; see notebook 03 for validation metrics and stability checks, and notebook 05 for an independent, exploratory check of a forced three-group solution.
- **Clinical boundary:** these are population-level lifestyle patterns. They must never be used to label, diagnose, or make judgements about any individual child.

## Save artifacts

In [ ]:
artifact = {
    "model_df": model_df,               # includes the "archetype" column
    "archetype_means_scaled": archetype_means_scaled,
    "FINAL_METHOD": FINAL_METHOD,
    "best_k": best_k,
}

with open(ARTIFACT_DIR / "04_archetype_characterisation.pkl", "wb") as f:
    pickle.dump(artifact, f)

print(f"Saved artifact -> {ARTIFACT_DIR / '04_archetype_characterisation.pkl'}")
